In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, re, time
import numpy as np, pandas as pd
from collections import Counter
ROOT = '/content/drive/MyDrive/geoproject'
os.makedirs(f'{ROOT}/raw_q', exist_ok=True)          # Phase Q raw harvests
os.makedirs(f'{ROOT}/results_q', exist_ok=True)
print('ready:', ROOT)

# Sprint Block M — Pakistan local track (M4a → M5), Q7, Bayesian EN, backbone ablation
Cell 0 mounts Drive and sets `ROOT`. Each phase below starts with a heading like this one; cells inside a phase are numbered in their first comment line (`# M4a.4 — …`). Every result is written to Drive under `results_m/`, `results_q/`, `results_p/`; nothing is hardcoded.

## Block M — M4a: ingest, validate and re-extract the 199-row Pakistan gold evaluation set
Input: `eval/pk_evalset_raw.json` (collection-time labels = annotator A). Output: `eval/pk_evalset_extracted.json` (same shape as `evalset_106_extracted.json`). The independent second annotation pass (annotator B = the author), Cohen's κ, and the adjudication of disagreements into the final `gold_tag` are done in the figures notebook (`sprint_block_l`, section L0); that notebook also re-scores every Pakistan arm on the adjudicated labels and rewrites the `results_m/` and `results_p/` Pakistan entries, so the Pakistan numbers printed by the cells below are the pre-adjudication run and the result files on Drive are authoritative.

In [ ]:
# M4a.0 — what is on Drive already
import os, glob, json
ROOT = '/content/drive/MyDrive/geoproject'
for p in sorted(glob.glob(f'{ROOT}/eval/*')): print(f'{os.path.getsize(p):>10,}  {p.split("/geoproject/")[1]}')
print('extractor:', os.path.exists(f'{ROOT}/adjective_extractor.py'))
print('results_m:', sorted(os.listdir(f'{ROOT}/results_m')) if os.path.isdir(f'{ROOT}/results_m') else None)
import subprocess; print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip())

In [ ]:
# M4a.1 — how Block H fetched GoogleNews W2V (read-only)
import json
NB = '/content/drive/MyDrive/Colab Notebooks'
H = json.load(open(f'{NB}/sprint_block_h.ipynb'))['cells']
for i in (16, 17):
    print(f'\n######## H[{i}] ########'); print(''.join(H[i]['source']))

In [ ]:
# M4a.2 — which PK gold tags fall outside the 227-lexicon (read-only)
import json
ROOT = '/content/drive/MyDrive/geoproject'
L = json.load(open(f'{ROOT}/eval/evalset_106.json'))
LEX, PLURAL = set(L['lexicon']), L['plural_map']
pk_tags = ['restaurant','cafe','park','museum','hotel','marketplace','mosque','fast-food','monument','bakery','shrine','ruins','mall','tomb','temple','fort','stadium','tower','city-gate','attraction','garden','zoo','college','bar','church','cemetery','gate','gallery','lake','beach','library','cinema','path','nightclub','cathedral','artwork','peak','retail','hill','cape','valley','village','hospital','planetarium','aquarium','bridge','square','building','office','cave-entrance','meadow','raceway','pond','residential']
out = [t for t in pk_tags if t not in LEX and PLURAL.get(t) not in LEX]
print('outside lexicon:', out)
import difflib
for t in out: print(f'  {t:15s} near:', difflib.get_close_matches(t, sorted(LEX), n=4, cutoff=0.5), '| contains:', [x for x in LEX if t.split("-")[0] in x][:6])
print(sorted(LEX))

In [ ]:
# M4a.3 — spaCy transformer model (same as H4)
import subprocess, sys, importlib.util
if importlib.util.find_spec('en_core_web_trf') is None:
    r = subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_trf', '-q'], capture_output=True, text=True)
    print(r.stdout[-800:], r.stderr[-800:])
import spacy, en_core_web_trf
print('spacy', spacy.__version__, '| gpu:', spacy.prefer_gpu())

In [ ]:
# M4a.4 — ingest, validate, re-extract the 199-row Pakistan gold eval set (H4 field mapping)
import json, sys, time, spacy
from collections import Counter
ROOT = '/content/drive/MyDrive/geoproject'
sys.path.insert(0, ROOT)
from adjective_extractor import PlaceInterpreter

LEX = set(json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon'])
rows = json.load(open(f'{ROOT}/eval/pk_evalset_raw.json'))
assert len(rows) == 199 and len({r['row'] for r in rows}) == 199, 'row count / id collision'
assert not [(r['row'], r['gold_tag']) for r in rows if r['gold_tag'] not in LEX], 'gold_tag outside lexicon'
print('alt_tags outside lexicon:', [(r['row'], a) for r in rows for a in r['alt_tags'] if a not in LEX])
for r in rows: r['lang'] = 'en'

spacy.prefer_gpu()
pi = PlaceInterpreter(model='en_core_web_trf')
t0 = time.time()
for r, it in zip(rows, pi.pipe([r['sentence'] for r in rows])):
    ours = [e['head_lemma'] for e in it.entities] + list(it.context_nouns)
    r['ours_nouns']  = sorted(set(ours))
    r['adjectives']  = sorted({a for e in it.entities for a in e['adjectives']})
    r['tag_hints']   = it.tag_hints
    r['transaction'] = it.transaction
print(f'extracted 199 in {time.time()-t0:.0f}s')
json.dump(rows, open(f'{ROOT}/eval/pk_evalset_extracted.json', 'w'), ensure_ascii=False, indent=1)

n_adj = sum(1 for r in rows if r['adjectives']); n_hint = sum(1 for r in rows if r['tag_hints'])
print(f'>=1 adjective: {n_adj}/199 ({n_adj/199:.1%}) | tag_hints fire: {n_hint}/199 | adj/row {sum(len(r["adjectives"]) for r in rows)/199:.2f}')
print('top adjectives:', Counter(a for r in rows for a in r['adjectives']).most_common(25))
print('gold tags n>=3:', len([t for t, c in Counter(r['gold_tag'] for r in rows).items() if c >= 3]), '| rows covered:', sum(c for c in Counter(r['gold_tag'] for r in rows).values() if c >= 3))
print(rows[0]['sentence']); print(rows[0]['ours_nouns'], rows[0]['adjectives'], rows[0]['transaction'])

In [ ]:
# M4a.5 — second annotation pass and agreement: done in the figures notebook (sprint_block_l, cells L0.1–L0.4).
# L0.1 writes a blind sheet (sentence only, shuffled) -> the author fills it -> L0.2 computes Cohen's kappa against the collection-time labels
# and stores the author's labels as `annotator_b_tag` -> L0.3/L0.4 write the disagreement sheet and apply the author's adjudication to `gold_tag`
# (the original label is kept as `collection_tag`). Summary of the current state of the evaluation file:
import json
from collections import Counter
pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json')); kap = json.load(open(f'{ROOT}/results_m/m4_kappa.json'))
d = Counter(r['gold_tag'] for r in pk); changed = sum(1 for r in pk if r.get('collection_tag', r['gold_tag']) != r['gold_tag'])
print(f"rows {len(pk)} | tags {len(d)} | n>=3 tags {sum(c >= 3 for c in d.values())} covering {sum(c for c in d.values() if c >= 3)} rows | kappa (A vs B) {kap['kappa_all']:.3f} | gold tags changed by adjudication {changed}")
print('protocol:', kap['protocol'])

## Block M — M5: region-specific experiments on the Pakistan gold set
Extends the W2V cache to the PK vocabulary, reloads the v1 ranker verbatim (Block I cells 17/18/22), recomputes both baselines on the PK set, and runs the transfer / local-only / merged / merged-PK×10 arms. Metrics on the n≥3 subset (158 rows, 23 tags) are primary. Writes `results_m/m5_transfer.json`, `m5_region_experiments.json`, `m5_en_local_merged.npz`.

In [ ]:
# M5.1 — extend the GoogleNews W2V subset cache with the PK eval vocabulary (H2 recipe; new file, v1 cache untouched)
import subprocess, sys, json, re, os, pickle, time, numpy as np
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gensim>=4.3.3'], check=True)
ROOT = '/content/drive/MyDrive/geoproject'
def parts(s): return [w for w in re.split(r'[\s\-_/]+', s) if w]
def variants(s):
    base = [s, s.lower(), s.title(), s.capitalize()]
    joined = [v.replace(' ', '_').replace('-', '_') for v in base]; squashed = [v.replace(' ', '').replace('-', '') for v in base]
    seen, out = set(), []
    for v in base + joined + squashed:
        if v and v not in seen: seen.add(v); out.append(v)
    return out
pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))
need = set()
for r in pk:
    for n in r['context_nouns'] + r['ours_nouns']: need.add(n); need.update(parts(n))
OLD = pickle.load(open(f'{ROOT}/models_cache/w2v_googlenews_subset.pkl', 'rb'))
probe = {v for s in need for v in variants(s)} - set(OLD)
print(f'{len(need)} PK strings -> {len(probe)} new surface forms to probe (old cache {len(OLD)})')
t0 = time.time()
import gensim.downloader as api
kv = api.load('word2vec-google-news-300')
NEW = {p: np.asarray(kv[p], dtype=np.float32) for p in probe if p in kv}
del kv; import gc; gc.collect()
VEC = {**OLD, **NEW}
out = f'{ROOT}/models_cache/w2v_googlenews_subset_v2.pkl'
pickle.dump(VEC, open(out, 'wb'), protocol=4)
hit = sum(1 for s in need if any(v in VEC for v in variants(s)) or any(v in VEC for w in parts(s) for v in variants(w)))
print(f'+{len(NEW)} vectors -> {len(VEC)} total, {os.path.getsize(out)/1e6:.1f} MB, {time.time()-t0:.0f}s | PK strings embeddable: {hit}/{len(need)}')
print('still OOV:', sorted(s for s in need if not (any(v in VEC for v in variants(s)) or any(v in VEC for w in parts(s) for v in variants(w))))[:40])

In [ ]:
# M5.2 — v1 ranker (verbatim Block I[17]/[18]/[22]) evaluated on the 199-row PK gold set: baselines, base-paper method, transfer arm
import json, re, pickle, numpy as np
from collections import Counter
from itertools import groupby
from sklearn.model_selection import KFold
ROOT = '/content/drive/MyDrive/geoproject'
LEX = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
W2V = pickle.load(open(f'{ROOT}/models_cache/w2v_googlenews_subset_v2.pkl', 'rb'))
PRUNE_K, CLUSTER_N, KS, LAMS = 20, 4, (1, 2, 3, 5, 10, 20), [0.0, 0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5]
ABSTRACT = {'place','spot','location','area','site','position','thing','kind','sort','lot','bit','way','day','time'}

def _parts(s): return [w for w in re.split(r'[\s\-_/]+', s) if w]
def _vars(s):
    b = [s, s.lower(), s.title(), s.capitalize()]; o, seen = [], set()
    for v in b + [x.replace(' ', '_').replace('-', '_') for x in b] + [x.replace(' ', '').replace('-', '') for x in b]:
        if v and v not in seen: seen.add(v); o.append(v)
    return o
def _wv(s):
    for v in _vars(s):
        if v in W2V: return W2V[v]
    sub = [W2V[v] for w in _parts(s) for v in _vars(w) if v in W2V][:8]
    return np.mean(sub, 0) if sub else None
W2TAGS = [t for t in LEX if _wv(t) is not None]
W2M = np.vstack([_wv(t) for t in W2TAGS]); W2M /= np.linalg.norm(W2M, axis=1, keepdims=True) + 1e-9
W2IX = {t: i for i, t in enumerate(W2TAGS)}
def w2v_score(terms):
    s = np.zeros(len(W2TAGS), np.float32)
    for cn in terms:
        v = _wv(cn)
        if v is None: continue
        sims = W2M @ (v / (np.linalg.norm(v) + 1e-9)); s += (sims - sims.min()) / (sims.max() - sims.min() + 1e-9)
    return s
def cluster_B(ranked, n=CLUSTER_N, k=PRUNE_K):
    pr = ranked[:k]; ps = set(pr); S = W2M[[W2IX[t] for t in pr]] @ W2M.T
    for i, t in enumerate(pr): S[i, W2IX[t]] = -np.inf
    votes = Counter({t: 0 for t in pr})
    for i in range(len(pr)):
        for j in np.argsort(-S[i])[:n]:
            if W2TAGS[j] in ps: votes[W2TAGS[j]] += 1
    cl, flat = [], []
    for _, g in groupby(sorted(votes.items(), key=lambda kv: -kv[1]), key=lambda kv: kv[1]):
        c = [t for t, _ in g]; cl.append(c); flat.extend(c)
    return cl, flat
def rank_of(gold, ranked, scores=None, clusters=None):
    if gold not in ranked: return len(ranked) + 1
    if clusters is not None:
        seen = 0
        for c in clusters:
            seen += len(c)
            if gold in c: return seen
    if scores is not None: return int((scores >= scores[W2IX[gold]]).sum())
    return ranked.index(gold) + 1
def metrics(ranks, ks=KS):
    r = np.asarray(ranks, float); o = {f'Hits@{k}': float((r <= k).mean()) for k in ks}
    o['nDCG@20'] = float(np.where(r <= 20, 1/np.log2(r+1), 0).mean()); o['MedR'] = float(np.median(r)); return o

def make_en(coef, classes, adj_vocab):
    aix = {a: j for j, a in enumerate(adj_vocab)}; c2w = {c: W2IX[c] for c in classes if c in W2IX}; cls = list(classes)
    def en_score(adjs):
        s = np.zeros(len(W2TAGS), np.float32); rows_c = [coef[:, aix[a]] for a in adjs if a in aix]
        if not rows_c: return s
        agg = np.sum(rows_c, axis=0)
        for c, w in c2w.items(): s[w] = agg[cls.index(c)]
        lo, hi = s.min(), s.max(); return (s - lo) / (hi - lo + 1e-9) if hi > lo else s
    return en_score
enz = np.load(f'{ROOT}/results_i/i3_elasticnet.npz', allow_pickle=True)
EN_V1 = make_en(enz['coef'], list(enz['classes']), list(enz['adj_vocab']))

pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))
gold = [r['gold_tag'] for r in pk]; dist = Counter(gold)
N3 = {t for t, c in dist.items() if c >= 3}; SUB = np.array([r['gold_tag'] in N3 for r in pk])
print(f'PK gold: {len(pk)} rows, {len(dist)} tags | n>=3 subset {SUB.sum()} rows / {len(N3)} tags | W2V-embeddable tags {len(W2TAGS)}/{len(LEX)}')
oov = Counter(t for r in pk for t in r['context_nouns'] if _wv(t) is None); print('OOV context nouns:', sum(oov.values()), oov.most_common(10))

def ranks_for(noun_key, lam, en, drop_abstract=True, clustered=False):
    out = []
    for r in pk:
        terms = [t for t in r[noun_key] if not (drop_abstract and t.lower() in ABSTRACT)]
        s = w2v_score(terms); s = (s - s.min()) / (s.max() - s.min() + 1e-9)
        if lam > 0 and en is not None: s = (1 - lam) * s + lam * en(r['adjectives'])
        ranked = [W2TAGS[i] for i in np.argsort(-s, kind='stable')]
        if clustered: cl, flat = cluster_B(ranked); out.append(rank_of(r['gold_tag'], flat, clusters=cl))
        else: out.append(rank_of(r['gold_tag'], ranked, scores=s))
    return np.array(out, float)
def cv_lam(noun_key, en):
    R = {L: ranks_for(noun_key, L, en) for L in LAMS}; idx = np.arange(len(pk)); held = np.zeros(len(pk)); ch = []
    for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(idx):
        L = max(LAMS, key=lambda l: (R[l][tr_i] <= 3).mean()); ch.append(L); held[te_i] = R[L][te_i]
    return held, ch, R

# baselines recomputed ON THE PK SET (worst-case ties)
maj = dist.most_common(1)[0][0]
RANKS = {'majority class': np.array([1.0 if g == maj else len(W2TAGS) + 1 for g in gold])}
RANKS['freq prior (oracle)'] = np.array([sum(c >= dist[g] for c in dist.values()) for g in gold], float)
# base-paper method (their Alg.2 / Alg.3) and H5's abstract-noun removal, on agent-A's context nouns
RANKS['W2V (Alg.2)']                 = ranks_for('context_nouns', 0, None, drop_abstract=False)
RANKS['W2V-Clustered (Alg.3)']       = ranks_for('context_nouns', 0, None, drop_abstract=False, clustered=True)
RANKS['W2V-Clust + abstract rm']     = ranks_for('context_nouns', 0, None, clustered=True)
RANKS['W2V plain + abstract rm']     = ranks_for('context_nouns', 0, None)
# transfer arm = v1 pipeline untouched
RANKS['transfer v1, λ=0.35 fixed']   = ranks_for('context_nouns', 0.35, EN_V1)
h, ch, R_tr = cv_lam('context_nouns', EN_V1); RANKS['transfer v1, λ by 5-fold CV'] = h; print('transfer CV folds chose λ =', ch)
# fully automatic variant: our extractor's nouns instead of agent-A's hand-picked context nouns
RANKS['[auto nouns] W2V plain']      = ranks_for('ours_nouns', 0, None)
RANKS['[auto nouns] transfer λ=0.35']= ranks_for('ours_nouns', 0.35, EN_V1)

def table(RANKS, mask=None, title=''):
    print(f'\n{title:-^100}'); print(f'{"arm":<32}' + ''.join(f'{f"H@{k}":>8}' for k in KS) + f'{"nDCG":>8}{"MedR":>7}')
    out = {}
    for name, rk in RANKS.items():
        m = metrics(rk if mask is None else rk[mask]); out[name] = m
        print(f'{name:<32}' + ''.join(f'{m[f"Hits@{k}"]:>8.3f}' for k in KS) + f'{m["nDCG@20"]:>8.3f}{m["MedR"]:>7.1f}')
    return out
RES = {'full_199': table(RANKS, None, ' PK gold — all 199 rows / 54 tags '), 'n3_158': table(RANKS, SUB, f' PK gold — n>=3 subset ({SUB.sum()} rows / {len(N3)} tags) — PRIMARY ')}
RES['transfer_cv_lambdas'] = ch; RES['transfer_lambda_curve_full'] = {str(l): metrics(R_tr[l]) for l in LAMS}
json.dump(RES, open(f'{ROOT}/results_m/m5_transfer.json', 'w'), indent=1); print('\nwrote results_m/m5_transfer.json')

In [ ]:
# M5.3 — local-only and merged arms: refit the frequentist EN (I3 recipe) on PK-only / global+PK weak-labelled rows, evaluate on the PK gold set
import time, pandas as pd, numpy as np, json
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
lab = pd.read_parquet(f'{ROOT}/results_m/weak_labelled_v2.parquet'); print(lab.columns.tolist(), len(lab))
def adjs_of(pj):
    p = json.loads(pj) if isinstance(pj, str) else pj
    return sorted({a.lower() for a, n, rel in p if rel == 'amod' and str(a).replace('-','').isalpha()})
lab['adjs'] = lab.pairs.apply(adjs_of)
trn = lab[lab.split == 'train'].reset_index(drop=True); pkt = trn[trn.region == 'pk']; glb = trn[trn.region != 'pk']
print(f'train: global {len(glb):,} | pk {len(pkt):,} ({pkt.weak_tag.nunique()} tags)')

def fit_en(df, vocab, tag='weak_tag'):
    aix = {a: j for j, a in enumerate(vocab)}; ri, ci = [], []
    for r, adjs in enumerate(df.adjs.values):
        for a in set(adjs):
            if a in aix: ri.append(r); ci.append(aix[a])
    X = sparse.csr_matrix((np.ones(len(ri), np.float32), (ri, ci)), shape=(len(df), len(vocab)))
    le = LabelEncoder().fit(df[tag]); y = le.transform(df[tag].values); t0 = time.time()
    en = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.5, max_iter=200, tol=1e-3, n_jobs=-1).fit(X, y)
    print(f'  EN fit {X.shape[0]:,} x {X.shape[1]} adj x {len(le.classes_)} tags, {time.time()-t0:.0f}s, {(en.coef_!=0).mean():.1%} nonzero')
    return make_en(en.coef_.astype(np.float32), list(le.classes_), list(vocab)), en.coef_, list(le.classes_)

V1_VOCAB = list(enz['adj_vocab'])
# local-only: 2.2k PK rows, adjective vocab = >=3 occurrences in PK train (data-starved by design)
c_pk = Counter(a for r in pkt.adjs for a in r); pk_vocab = sorted(a for a, c in c_pk.items() if c >= 3)
print(f'local-only vocab {len(pk_vocab)} adjectives | PK-eval adjective coverage {np.mean([any(a in set(pk_vocab) for a in r["adjectives"]) for r in pk]):.0%}')
# merged: I3's stratified 150k global subsample (random_state=42) + all PK train rows, v1 vocab; PK x10 = PK rows duplicated 10x
# (same objective as sample_weight=10, but saga keeps its normal step size — the weighted fit did not converge in 40 min)
keep = (glb.groupby('weak_tag', group_keys=False).apply(lambda g: g.sample(min(len(g), max(1, round(len(g) * 150_000 / len(glb)))), random_state=42)).index.values)
merged = pd.concat([glb.loc[keep], pkt]).reset_index(drop=True)
if 'EN_LOC' not in dir(): EN_LOC, C_loc, cls_loc = fit_en(pkt, pk_vocab)
if 'EN_MRG' not in dir(): EN_MRG, C_mrg, _ = fit_en(merged, V1_VOCAB)
EN_M10, C_m10, _ = fit_en(pd.concat([glb.loc[keep]] + [pkt] * 10).reset_index(drop=True), V1_VOCAB)

for name, en in [('local-only', EN_LOC), ('merged global+PK', EN_MRG), ('merged, PK x10', EN_M10)]:
    RANKS[f'{name}, λ=0.35']            = ranks_for('context_nouns', 0.35, en)
    h, ch, _ = cv_lam('context_nouns', en); RANKS[f'{name}, λ by CV'] = h; print(f'{name}: CV folds chose λ = {ch}')
RES = {'full_199': table(RANKS, None, ' PK gold — all 199 rows / 54 tags '), 'n3_158': table(RANKS, SUB, f' PK gold — n>=3 subset ({SUB.sum()} rows / {len(N3)} tags) — PRIMARY ')}
np.savez(f'{ROOT}/results_m/m5_en_local_merged.npz', coef_local=C_loc, classes_local=np.array(cls_loc), vocab_local=np.array(pk_vocab), coef_merged=C_mrg, coef_merged_pk10=C_m10, vocab_v1=np.array(V1_VOCAB))
json.dump(RES, open(f'{ROOT}/results_m/m5_region_experiments.json', 'w'), indent=1); print('\nwrote results_m/m5_region_experiments.json')

## Block Q — Q7: v1 + filtered Wikidata (Option B)
One number: does targeted Wikidata augmentation of the tags Wikivoyage is thin on beat v1 (Hits@3 .509) on the 106 — and does it transfer to the PK set? Outcome: +3 rows on the 106, −4 rows on PK → **v1 stays the pipeline.** Writes `results_q/q7_v1_vs_v2.json`, `q7_v2_en.npz`, `results_m/m5_transfer_v2.json`.

In [ ]:
# Q7 — Option B: v1 (Wikivoyage) + FILTERED Wikidata augmentation of the tags Wikivoyage is thin on. One number: beats v1 Hits@3 0.509 on the 106?
# Feature set = I3's adjective-level EN (the recipe behind the .509 headline), so the comparison is exact.
import time, json, numpy as np, pandas as pd
from collections import Counter
from sklearn.model_selection import KFold
cv3 = pd.read_parquet(f'{ROOT}/annotated/corpus_v3.parquet', columns=['label', 'label_source', 'split', 'pairs'])
wd = cv3[(cv3.label_source == 'wikidata_p31') & (cv3.split == 'train')].copy(); del cv3
wd['adjs'] = wd.pairs.apply(adjs_of); wd = wd[wd.adjs.str.len() > 0]
wv_cnt, wd_cnt = trn.weak_tag.value_counts(), wd.label.value_counts()
THIN, MIN_WD, CAP = 3000, 50, 5000
aug_tags = sorted(t for t in wd_cnt.index if wv_cnt.get(t, 0) < THIN and wd_cnt[t] >= MIN_WD)
print(f'Wikidata adjective-bearing train rows: {len(wd):,} | tags where Wikivoyage train < {THIN} and Wikidata has >= {MIN_WD}: {len(aug_tags)}')
print('   ' + ', '.join(f'{t}(wv {wv_cnt.get(t,0)}, wd {wd_cnt[t]})' for t in aug_tags))
aug = pd.concat([wd[wd.label == t].sample(min(CAP, wd_cnt[t]), random_state=42) for t in aug_tags])
aug = aug.rename(columns={'label': 'weak_tag'})[['weak_tag', 'adjs']].assign(region='global')
v2 = pd.concat([trn[['weak_tag', 'adjs', 'region']], aug]).reset_index(drop=True)
print(f'corpus_v2 train: {len(v2):,} = {len(trn):,} Wikivoyage + {len(aug):,} filtered Wikidata')
V2_VOCAB = sorted(a for a, c in Counter(a for r in v2.adjs for a in r).items() if c >= 30)
ev106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json'))
print(f'v2 adjective vocab {len(V2_VOCAB)} (v1: {len(V1_VOCAB)}) | 106-eval adjective coverage {np.mean([any(a in set(V2_VOCAB) for a in r["adjectives"]) for r in ev106]):.0%}')
keep2 = (v2.groupby('weak_tag', group_keys=False).apply(lambda g: g.sample(min(len(g), max(1, round(len(g) * 150_000 / len(v2)))), random_state=42)).index.values)
EN_V2, C_v2, cls_v2 = fit_en(v2.loc[keep2], V2_VOCAB)

def ranks_106(lam, en):
    out = []
    for r in ev106:
        terms = [t for t in r['context_nouns'] if t.lower() not in ABSTRACT]
        s = w2v_score(terms); s = (s - s.min()) / (s.max() - s.min() + 1e-9)
        if lam > 0: s = (1 - lam) * s + lam * en(r['adjectives'])
        ranked = [W2TAGS[i] for i in np.argsort(-s, kind='stable')]; out.append(rank_of(r['gold_tag'], ranked, scores=s))
    return np.array(out, float)
def cv_106(en):
    R = {L: ranks_106(L, en) for L in LAMS}; idx = np.arange(len(ev106)); held = np.zeros(len(ev106)); ch = []
    for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(idx):
        L = max(LAMS, key=lambda l: (R[l][tr_i] <= 3).mean()); ch.append(L); held[te_i] = R[L][te_i]
    return metrics(held), ch, {str(l): metrics(R[l]) for l in LAMS}
m_v1, ch1, curve1 = cv_106(EN_V1); m_v2, ch2, curve2 = cv_106(EN_V2)
print(f'\n{"Q7 — 106 eval, 5-fold CV λ":-^70}')
for name, m, ch in [('v1 Wikivoyage-only (J1 re-run)', m_v1, ch1), ('v2 = v1 + filtered Wikidata', m_v2, ch2)]:
    print(f'  {name:<32} λ{ch}  H@1 {m["Hits@1"]:.3f}  H@3 {m["Hits@3"]:.3f}  H@5 {m["Hits@5"]:.3f}  nDCG {m["nDCG@20"]:.3f}')
print(f'  verdict: v2 {"BEATS" if m_v2["Hits@3"] > m_v1["Hits@3"] else "does NOT beat"} v1 on Hits@3 ({m_v2["Hits@3"]:.3f} vs {m_v1["Hits@3"]:.3f})')
np.savez(f'{ROOT}/results_q/q7_v2_en.npz', coef=C_v2, classes=np.array(cls_v2), adj_vocab=np.array(V2_VOCAB))
json.dump({'aug_tags': aug_tags, 'aug_rows': int(len(aug)), 'thresholds': {'wv_thin': THIN, 'wd_min': MIN_WD, 'cap': CAP},
           'v1_cv': m_v1, 'v1_folds': ch1, 'v1_curve': curve1, 'v2_cv': m_v2, 'v2_folds': ch2, 'v2_curve': curve2},
          open(f'{ROOT}/results_q/q7_v1_vs_v2.json', 'w'), indent=1); print('wrote results_q/q7_v1_vs_v2.json')

In [ ]:
# Q7b — does the v2 EN transfer to the PK gold set as well as v1? (no refit; same PK protocol as M5.2)
R2 = {'transfer v1, λ=0.35': RANKS['transfer v1, λ=0.35 fixed'], 'transfer v2, λ=0.35': ranks_for('context_nouns', 0.35, EN_V2)}
h, ch, _ = cv_lam('context_nouns', EN_V2); R2['transfer v2, λ by CV'] = h; print('v2 transfer CV folds chose λ =', ch)
R2['[auto nouns] transfer v2 λ=0.35'] = ranks_for('ours_nouns', 0.35, EN_V2)
out = {'full_199': table(R2, None, ' PK gold — all 199 '), 'n3_158': table(R2, SUB, ' PK gold — n>=3 subset (PRIMARY) ')}
json.dump(out, open(f'{ROOT}/results_m/m5_transfer_v2.json', 'w'), indent=1); print('wrote results_m/m5_transfer_v2.json')

## Block P Part 1 — Bayesian Elastic Net on the 966 FP-Growth patterns (P1.4 – P1.7b)
Full posterior over pattern → tag weights (Li & Lin 2010 Bayesian elastic net, scale-mixture form), NUTS on GPU. Compared with the frequentist EN (P1.3), evaluated as a blend on the 106 + PK sets, and Figure 2 (credible non-circular patterns with 94% HDIs). Ran on the A100. Writes `results_p/p1_bayesian_en.npz`, `p1_svi_en.npz`, `p1_nuts_check.npz`, `p1_ben_figure2.json`.

In [ ]:
# P1.4 prep — inspect the P1 artifacts (read-only)
import numpy as np, os
from scipy import sparse
for f in sorted(os.listdir(f'{ROOT}/results_p')): print(f'{os.path.getsize(f"{ROOT}/results_p/{f}")/1e6:8.1f} MB  {f}')
meta = np.load(f'{ROOT}/results_p/p1_feat_meta.npz', allow_pickle=True); print('meta keys:', {k: (meta[k].shape, meta[k].dtype) for k in meta.files})
fen = np.load(f'{ROOT}/results_p/p1_frequentist_en.npz', allow_pickle=True); print('fen keys:', {k: meta and (fen[k].shape, fen[k].dtype) for k in fen.files})
Xtr = sparse.load_npz(f'{ROOT}/results_p/Xtr.npz'); Xev = sparse.load_npz(f'{ROOT}/results_p/Xev.npz')
print('Xtr', Xtr.shape, f'nnz/row {Xtr.getnnz(1).mean():.2f}, rows with >=1 pattern {(Xtr.getnnz(1)>0).mean():.0%} | Xev', Xev.shape, f'{(Xev.getnnz(1)>0).mean():.0%} covered')
pv = list(meta['patvocab']) if 'patvocab' in meta.files else list(fen['patvocab']); print(len(pv), 'patterns; first/last:', pv[:5], pv[-3:], '| n_adj', int(meta['n_adj']) if 'n_adj' in meta.files else '?')
import jax; print('jax', jax.__version__, 'backend', jax.default_backend(), jax.devices())

In [ ]:
# P1.4 — Bayesian Elastic Net on v1's 966 FP-Growth patterns, full-batch SVI on the A100 (numpyro, GPU jax kept)
import subprocess, sys, time, json, numpy as np
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'numpyro', 'multipledispatch'], check=True)
import jax, jax.numpy as jnp, numpyro, numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO
from numpyro.infer.autoguide import AutoNormal
assert jax.default_backend() == 'gpu', jax.default_backend()
from scipy import sparse
Xtr = sparse.load_npz(f'{ROOT}/results_p/Xtr.npz'); meta = np.load(f'{ROOT}/results_p/p1_feat_meta.npz', allow_pickle=True)
PV, CLASSES, ytr = list(meta['patvocab']), list(meta['classes']), meta['ytr']; P, C = len(PV), len(CLASSES)
# stratified 80k subsample of rows that carry >=1 pattern (rows with none only inform the intercept)
rng = np.random.default_rng(42); has = np.where(Xtr.getnnz(1) > 0)[0]; N_SUB = 80_000
per = {c: max(1, round(N_SUB * (ytr[has] == c).mean())) for c in range(C)}
sub = np.concatenate([rng.choice(has[ytr[has] == c], min(per[c], (ytr[has] == c).sum()), replace=False) for c in range(C)])
X = jnp.asarray(Xtr[sub].toarray(), jnp.float32); y = jnp.asarray(ytr[sub]); N = len(sub)
print(f'BEN data: {N:,} x {P} patterns x {C} tags on {jax.devices()[0]}')

LAM2 = 0.5   # ridge share of the frequentist objective (l1_ratio=0.5, C=0.5) — fixed; the Laplace scale b is learned
def model(X, y=None):
    b = numpyro.sample('b', dist.HalfNormal(1.0))
    beta = numpyro.sample('beta', dist.Laplace(0.0, b).expand([P, C]).to_event(2))
    numpyro.factor('ridge', -LAM2 * jnp.sum(beta ** 2))          # Laplace x Gaussian = Bayesian elastic net prior (Li & Lin 2010)
    alpha = numpyro.sample('alpha', dist.Normal(0.0, 2.0).expand([C]).to_event(1))
    with numpyro.plate('n', X.shape[0]):
        numpyro.sample('y', dist.Categorical(logits=X @ beta + alpha), obs=y)

guide = AutoNormal(model); svi = SVI(model, guide, numpyro.optim.Adam(0.01), Trace_ELBO())
t0 = time.time(); state = svi.init(jax.random.PRNGKey(0), X, y)
step = jax.jit(svi.update); losses = []
for i in range(6000):
    state, loss = step(state, X, y); losses.append(float(loss)) if i % 200 == 0 else None
ben_time = time.time() - t0
params = svi.get_params(state)
mu, sd = np.asarray(params['beta_auto_loc']), np.asarray(jnp.exp(params['beta_auto_scale']) if 'beta_auto_scale' in params else params['beta_auto_scale'])
print(f'SVI 6000 full-batch steps in {ben_time:.0f}s | ELBO trace: {[round(l/N,3) for l in losses[::5]]}')
lo, hi = mu - 1.881 * sd, mu + 1.881 * sd                        # 94% interval of the mean-field normal posterior
SEL = (lo > 0) | (hi < 0)
print(f'posterior: {SEL.mean():.1%} of {P*C:,} (pattern,tag) coefficients have 94% HDI excluding 0 | learned Laplace scale b = {float(jnp.exp(params["b_auto_loc"])) if "b_auto_loc" in params else "?"}')

# ---- compare with the frequentist EN on the same patterns (P1.3) ----
from scipy.stats import spearmanr
fen = np.load(f'{ROOT}/results_p/p1_frequentist_en.npz', allow_pickle=True); FC = fen['coef'].T   # (P, C)
assert list(fen['patvocab']) == PV and list(fen['classes']) == CLASSES
sig_f, sig_b = np.abs(FC).max(1), np.abs(mu * SEL).max(1)
rho = spearmanr(sig_f, sig_b).correlation
sel_f, sel_b = set(np.where(sig_f > 0)[0]), set(np.where(sig_b > 0)[0])
print(f'frequentist {len(sel_f)} patterns selected ({fen["fit_time_s"]:.0f}s) | Bayesian {len(sel_b)} ({ben_time:.0f}s) | Jaccard {len(sel_f & sel_b)/len(sel_f | sel_b):.2f} | Spearman rho(|coef| rankings) {rho:.3f}')
top = np.argsort(-sig_b)[:30]
print('top Bayesian patterns -> strongest tag [posterior mean, 94% HDI]:')
for j in top[:20]:
    k = int(np.argmax(np.abs(mu[j]))); print(f'   {PV[j]:<26} -> {CLASSES[k]:<12} {mu[j,k]:+.2f} [{lo[j,k]:+.2f}, {hi[j,k]:+.2f}]')
np.savez(f'{ROOT}/results_p/p1_bayesian_en.npz', mean=mu, sd=sd, hdi_lo=lo, hdi_hi=hi, selected=SEL, patvocab=np.array(PV), classes=np.array(CLASSES),
         n_sub=N, steps=6000, lam2=LAM2, fit_time_s=ben_time, method='numpyro AutoNormal full-batch SVI, GPU')
json.dump({'figure2': [{'pattern': PV[j], 'tag': CLASSES[int(np.argmax(np.abs(mu[j])))], 'mean': float(mu[j, int(np.argmax(np.abs(mu[j])))]),
                        'hdi': [float(lo[j, int(np.argmax(np.abs(mu[j])))]), float(hi[j, int(np.argmax(np.abs(mu[j])))])], 'freq_coef': float(FC[j, int(np.argmax(np.abs(mu[j])))])} for j in top],
           'n_selected_bayes': int(len(sel_b)), 'n_selected_freq': int(len(sel_f)), 'jaccard': len(sel_f & sel_b)/len(sel_f | sel_b), 'spearman_rho': float(rho),
           'time_bayes_s': ben_time, 'time_freq_s': float(fen['fit_time_s']), 'pct_hdi_excl_0': float(SEL.mean())},
          open(f'{ROOT}/results_p/p1_ben_figure2.json', 'w'), indent=1)
print('wrote results_p/p1_bayesian_en.npz, p1_ben_figure2.json')

In [ ]:
# P1.5 — pattern-level blends on the 106 + PK: frequentist EN (P1.3) vs Bayesian EN (NUTS posterior) — and the non-circular Figure 2
import json, numpy as np
from sklearn.model_selection import KFold
from scipy.stats import spearmanr
ben = np.load(f'{ROOT}/results_p/p1_bayesian_en.npz', allow_pickle=True); MU, SEL = ben['mean'], ben['selected']
fen = np.load(f'{ROOT}/results_p/p1_frequentist_en.npz', allow_pickle=True); FC = fen['coef'].T
ADJ_SET, PAIR_SET = set(PV[:int(meta['n_adj'])]), set(PV[int(meta['n_adj']):]); PIX = {p: j for j, p in enumerate(PV)}
def pat_feats(r): return [PIX[a] for a in r['adjectives'] if a in ADJ_SET] + [PIX[p] for p in r['transaction'] if p in PAIR_SET]
def make_pen(coefPC):   # (P, C) pattern->tag weights -> en_score over the sentence's active patterns, aligned to W2TAGS
    c2w = {c: W2IX[c] for c in CLASSES if c in W2IX}; cix = {c: k for k, c in enumerate(CLASSES)}
    def en(r):
        s = np.zeros(len(W2TAGS), np.float32); f = pat_feats(r)
        if not f: return s
        agg = coefPC[f].sum(0)
        for c, w in c2w.items(): s[w] = agg[cix[c]]
        lo, hi = s.min(), s.max(); return (s - lo) / (hi - lo + 1e-9) if hi > lo else s
    return en
ENS = {'freq EN (966 patterns)': make_pen(FC), 'Bayesian EN, HDI-selected': make_pen(MU * SEL), 'Bayesian EN, posterior mean': make_pen(MU)}

# frequentist vs Bayesian selection agreement
sig_f, sig_b = np.abs(FC).max(1), np.abs(MU * SEL).max(1); sel_f, sel_b = set(np.where(sig_f > 0)[0]), set(np.where(sig_b > 0)[0])
rho = spearmanr(sig_f, np.abs(MU).max(1)).correlation
print(f'selection: frequentist {len(sel_f)} patterns nonzero ({float(fen["fit_time_s"]):.0f}s) | Bayesian {len(sel_b)} credible ({float(ben["fit_time_s"]):.0f}s NUTS, {ben["method"]}) | Jaccard {len(sel_f & sel_b)/len(sel_f | sel_b):.2f} | Spearman rho(|coef|) {rho:.3f}')

def rk(rows, lam, en, noun_key='context_nouns'):
    out = []
    for r in rows:
        terms = [t for t in r[noun_key] if t.lower() not in ABSTRACT]; s = w2v_score(terms); s = (s - s.min()) / (s.max() - s.min() + 1e-9)
        if lam > 0: s = (1 - lam) * s + lam * en(r)
        ranked = [W2TAGS[i] for i in np.argsort(-s, kind='stable')]; out.append(rank_of(r['gold_tag'], ranked, scores=s))
    return np.array(out, float)
def cv(rows, en):
    R = {L: rk(rows, L, en) for L in LAMS}; held = np.zeros(len(rows)); ch = []
    for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(np.arange(len(rows))):
        L = max(LAMS, key=lambda l: (R[l][tr_i] <= 3).mean()); ch.append(L); held[te_i] = R[L][te_i]
    return held, ch, R
OUT = {}
for setname, rows, mask in [('106', ev106, None), ('PK n>=3', pk, SUB)]:
    print(f'\n{f" {setname} — pattern-level blends, λ by 5-fold CV ":-^84}')
    print(f'  {"arm":<32}{"λ":>6}{"H@1":>8}{"H@3":>8}{"H@5":>8}{"nDCG":>8}')
    base = rk(rows, 0, None); m = metrics(base if mask is None else base[mask]); print(f'  {"W2V nouns-only":<32}{"-":>6}{m["Hits@1"]:>8.3f}{m["Hits@3"]:>8.3f}{m["Hits@5"]:>8.3f}{m["nDCG@20"]:>8.3f}')
    OUT[setname] = {'nouns_only': m}
    for name, en in ENS.items():
        held, ch, R = cv(rows, en); m = metrics(held if mask is None else held[mask]); m35 = metrics(R[0.35] if mask is None else R[0.35][mask])
        print(f'  {name:<32}{np.median(ch):>6.2f}{m["Hits@1"]:>8.3f}{m["Hits@3"]:>8.3f}{m["Hits@5"]:>8.3f}{m["nDCG@20"]:>8.3f}   (λ=.35: H@3 {m35["Hits@3"]:.3f})')
        OUT[setname][name] = {'cv': m, 'folds': ch, 'lam035': m35}
print('\n  reference — adjective-level EN (I3/J1) on the 106: H@3 .509; transfer to PK n>=3: .589')

# Figure 2: top credible NON-circular patterns (J5 rule: drop rows whose pattern text contains the tag word)
def circular(p, t): w = p.split(':', 1)[-1].replace('_', ' '); return t in w or t.replace('-', ' ') in w
fig = []
for j in np.argsort(-sig_b):
    if sig_b[j] == 0 or len(fig) >= 30: break
    k = int(np.argmax(np.abs(MU[j] * SEL[j]))); t = CLASSES[k]
    if circular(PV[j], t): continue
    fig.append({'pattern': PV[j], 'tag': t, 'mean': float(MU[j, k]), 'hdi': [float(ben['hdi_lo'][j, k]), float(ben['hdi_hi'][j, k])], 'freq_coef': float(FC[j, k])})
print('\nFigure 2 — top credible non-circular pattern -> tag [posterior mean, 94% HDI] (freq coef):')
for f in fig[:25]: print(f'   {f["pattern"]:<26} -> {f["tag"]:<12} {f["mean"]:+.2f} [{f["hdi"][0]:+.2f}, {f["hdi"][1]:+.2f}]   ({f["freq_coef"]:+.2f})')
json.dump({'method': str(ben['method']), 'n_sub': int(ben['n_sub']), 'chains': int(ben['chains']), 'draws': int(ben['draws']), 'b_mean': float(ben['b_mean']), 'divergences': int(ben['divergences']),
           'rhat_top200_median': float(ben['rhat_top200_median']), 'rhat_top200_max': float(ben['rhat_top200_max']), 'ess_top200_median': float(ben['ess_top200_median']),
           'n_selected_bayes': len(sel_b), 'n_selected_freq': len(sel_f), 'jaccard': len(sel_f & sel_b)/len(sel_f | sel_b), 'spearman_rho': float(rho), 'pct_credible': float(SEL.mean()),
           'time_bayes_s': float(ben['fit_time_s']), 'time_freq_s': float(fen['fit_time_s']), 'figure2_noncircular': fig, 'eval': OUT},
          open(f'{ROOT}/results_p/p1_ben_figure2.json', 'w'), indent=1); print('\nwrote results_p/p1_ben_figure2.json')

In [ ]:
# P1.6 — NUTS check of the SVI posterior (same model, 20k stratified rows, GPU). Reports agreement with the SVI means + HDI widths.
import time, numpy as np, jax, jax.numpy as jnp, numpyro
from numpyro.infer import MCMC, NUTS
from scipy.stats import spearmanr
rng = np.random.default_rng(1); N_N = 20_000
per = {c: max(1, round(N_N * (ytr[has] == c).mean())) for c in range(C)}
sub_n = np.concatenate([rng.choice(has[ytr[has] == c], min(per[c], (ytr[has] == c).sum()), replace=False) for c in range(C)])
Xn, yn = jnp.asarray(Xtr[sub_n].toarray(), jnp.float32), jnp.asarray(ytr[sub_n])
t0 = time.time()
mcmc = MCMC(NUTS(model, max_tree_depth=8, target_accept_prob=0.8), num_warmup=300, num_samples=300, progress_bar=False)
mcmc.run(jax.random.PRNGKey(0), Xn, yn)
nuts_time = time.time() - t0
S = mcmc.get_samples(); B = np.asarray(S['beta'])                     # (300, P, C)
mu_n, lo_n, hi_n = B.mean(0), np.percentile(B, 3, axis=0), np.percentile(B, 97, axis=0)
sel_n = (lo_n > 0) | (hi_n < 0)
ext = mcmc.get_extra_fields(); div = int(np.asarray(ext['diverging']).sum()) if 'diverging' in ext else -1
print(f'NUTS {N_N:,} rows, 300+300 draws in {nuts_time:.0f}s | divergences {div} | learned b: NUTS {float(np.asarray(S["b"]).mean()):.3f} vs SVI {float(jnp.exp(params["b_auto_loc"])):.3f}')
flat = lambda a: a.reshape(-1)
print(f'posterior means, SVI vs NUTS: Pearson r {np.corrcoef(flat(MU), flat(mu_n))[0,1]:.3f} | Spearman rho on |mean| per pattern {spearmanr(np.abs(MU).max(1), np.abs(mu_n).max(1)).correlation:.3f}')
print(f'94% HDI width: SVI median {np.median(ben["hdi_hi"] - ben["hdi_lo"]):.2f} vs NUTS {np.median(hi_n - lo_n):.2f} | credible coefficients: SVI {SEL.mean():.2%} vs NUTS {sel_n.mean():.2%} | Jaccard of credible sets {(SEL & sel_n).sum() / max(1, (SEL | sel_n).sum()):.2f}')
top_n = [(PV[j], CLASSES[int(np.argmax(np.abs(mu_n[j])))]) for j in np.argsort(-np.abs(mu_n).max(1))[:12]]; print('NUTS top patterns:', top_n)
np.savez(f'{ROOT}/results_p/p1_nuts_check.npz', mean=mu_n, hdi_lo=lo_n, hdi_hi=hi_n, selected=sel_n, n_sub=N_N, time_s=nuts_time, divergences=div)
print('wrote results_p/p1_nuts_check.npz')

In [ ]:
# P1.7 — NUTS as THE Bayesian Elastic Net fit: full 80k stratified subsample, 500 warmup + 500 draws, GPU. SVI result kept aside as the fast approximation.
import time, os, numpy as np, jax, jax.numpy as jnp
from numpyro.infer import MCMC, NUTS
os.replace(f'{ROOT}/results_p/p1_bayesian_en.npz', f'{ROOT}/results_p/p1_svi_en.npz')
t0 = time.time()
mcmc = MCMC(NUTS(model, max_tree_depth=8, target_accept_prob=0.8), num_warmup=500, num_samples=500, progress_bar=False)
mcmc.run(jax.random.PRNGKey(0), X, y, extra_fields=('diverging', 'num_steps'))
nuts_time = time.time() - t0
S = mcmc.get_samples(); B = np.asarray(S['beta']); ex = mcmc.get_extra_fields()
mu_n, lo_n, hi_n = B.mean(0), np.percentile(B, 3, axis=0), np.percentile(B, 97, axis=0); sel_n = (lo_n > 0) | (hi_n < 0)
# split-R-hat on the single chain (first/second half) as a quick mixing check for the top coefficients
half = B.shape[0] // 2; m1, m2 = B[:half].mean(0), B[half:].mean(0); v1, v2 = B[:half].var(0, ddof=1), B[half:].var(0, ddof=1)
W = (v1 + v2) / 2; Bv = half * ((m1 - m2) ** 2) / 2; rhat = np.sqrt((W * (half - 1) / half + Bv / half) / (W + 1e-12))
top = np.argsort(-np.abs(mu_n).max(1))[:200]
print(f'NUTS {N:,} rows x {P} x {C}: {nuts_time:.0f}s | divergences {int(np.asarray(ex["diverging"]).sum())} | mean leapfrog steps {float(np.asarray(ex["num_steps"]).mean()):.0f} | b = {float(np.asarray(S["b"]).mean()):.3f}')
print(f'split-Rhat over top-200 patterns (max over tags): median {np.median(rhat[top].max(1)):.3f}, max {rhat[top].max():.3f} | credible coefficients {sel_n.mean():.2%} ({sel_n.any(1).sum()} patterns)')
np.savez(f'{ROOT}/results_p/p1_bayesian_en.npz', mean=mu_n, sd=B.std(0), hdi_lo=lo_n, hdi_hi=hi_n, selected=sel_n, patvocab=np.array(PV), classes=np.array(CLASSES),
         n_sub=N, warmup=500, draws=500, lam2=LAM2, fit_time_s=nuts_time, method='numpyro NUTS, GPU, max_tree_depth 8', b_mean=float(np.asarray(S['b']).mean()),
         divergences=int(np.asarray(ex['diverging']).sum()), rhat_top200_max=float(rhat[top].max()))
print('wrote results_p/p1_bayesian_en.npz (NUTS); SVI fit moved to p1_svi_en.npz')

In [ ]:
# P1.7b — Bayesian Elastic Net in its scale-mixture (Li & Lin 2010) form: beta ~ N(0, tau^2), tau^2 ~ Exp(1/(2 b^2)) => marginal Laplace(0, b); ridge factor exp(-lam2 beta^2).
# Non-centred, smooth => HMC-friendly. 2 vectorised NUTS chains on the 80k subsample, GPU.
import time, numpy as np, jax, jax.numpy as jnp, numpyro, numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from numpyro.diagnostics import gelman_rubin, effective_sample_size
def model_sm(X, y=None):
    b = numpyro.sample('b', dist.HalfNormal(1.0))
    tau2 = numpyro.sample('tau2', dist.Exponential(1.0 / (2.0 * b ** 2)).expand([P, C]).to_event(2))
    z = numpyro.sample('z', dist.Normal(0.0, 1.0).expand([P, C]).to_event(2))
    beta = numpyro.deterministic('beta', z * jnp.sqrt(tau2))
    numpyro.factor('ridge', -LAM2 * jnp.sum(beta ** 2))
    alpha = numpyro.sample('alpha', dist.Normal(0.0, 2.0).expand([C]).to_event(1))
    with numpyro.plate('n', X.shape[0]):
        numpyro.sample('y', dist.Categorical(logits=X @ beta + alpha), obs=y)
t0 = time.time()
mcmc = MCMC(NUTS(model_sm, max_tree_depth=10, target_accept_prob=0.8), num_warmup=500, num_samples=500, num_chains=2, chain_method='vectorized', progress_bar=False)
mcmc.run(jax.random.PRNGKey(0), X, y, extra_fields=('diverging', 'num_steps'))
nuts_time = time.time() - t0
S = mcmc.get_samples(group_by_chain=True); ex = mcmc.get_extra_fields()
B = np.asarray(S['beta'])                                         # (2, 500, P, C)
top = np.argsort(-np.abs(B.mean((0, 1))).max(1))[:200]
rh = np.asarray(gelman_rubin(B[:, :, top, :])); ess = np.asarray(effective_sample_size(B[:, :, top, :]))
Bf = B.reshape(-1, P, C); mu_n, lo_n, hi_n = Bf.mean(0), np.percentile(Bf, 3, axis=0), np.percentile(Bf, 97, axis=0); sel_n = (lo_n > 0) | (hi_n < 0)
print(f'NUTS (scale-mixture) 2 chains x 500+500 on {N:,} rows: {nuts_time:.0f}s | divergences {int(np.asarray(ex["diverging"]).sum())} | mean leapfrog steps {float(np.asarray(ex["num_steps"]).mean()):.0f} | b = {float(np.asarray(S["b"]).mean()):.3f}')
print(f'R-hat over top-200 patterns x 69 tags: median {np.median(rh):.3f}, 95th pct {np.percentile(rh, 95):.3f}, max {rh.max():.3f} | ESS median {np.median(ess):.0f} | credible coefficients {sel_n.mean():.2%} ({sel_n.any(1).sum()} patterns) | median 94% HDI width {np.median(hi_n - lo_n):.2f}')
np.savez(f'{ROOT}/results_p/p1_bayesian_en.npz', mean=mu_n, sd=Bf.std(0), hdi_lo=lo_n, hdi_hi=hi_n, selected=sel_n, patvocab=np.array(PV), classes=np.array(CLASSES),
         n_sub=N, warmup=500, draws=500, chains=2, lam2=LAM2, fit_time_s=nuts_time, method='numpyro NUTS, scale-mixture BEN (Li & Lin 2010), non-centred, GPU',
         b_mean=float(np.asarray(S['b']).mean()), divergences=int(np.asarray(ex['diverging']).sum()), rhat_top200_median=float(np.median(rh)), rhat_top200_max=float(rh.max()), ess_top200_median=float(np.median(ess)))
print('wrote results_p/p1_bayesian_en.npz (NUTS scale-mixture)')

## Block P Part 2 — Semantic backbone ablation
7 frozen backbones (W2V, GloVe, fastText, ConceptNet Numberbatch, all-mpnet-base-v2, bge-base-en-v1.5, multilingual-e5-base) × 3 conditions (nouns-only / + all adjectives naive / + Elastic-Net blend), on the 106 and the PK gold set; then the fine-tuned-classifier arm and the pattern-granularity arm. Runs on a T4. Writes `results_p/p2_*`.

In [ ]:
# P2.1 — probe vocabulary + static-vector subset caches (GloVe, fastText, ConceptNet Numberbatch) via gensim-data; H2 recipe, cached to Drive
import json, re, os, pickle, time, gc, subprocess, sys, numpy as np
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gensim>=4.3.3'], check=True)
ROOT = '/content/drive/MyDrive/geoproject'
ev106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json'))
def parts(s): return [w for w in re.split(r'[\s\-_/]+', s) if w]
def variants(s):
    base = [s, s.lower(), s.title(), s.capitalize()]; joined = [v.replace(' ', '_').replace('-', '_') for v in base]; squashed = [v.replace(' ', '').replace('-', '') for v in base]
    seen, out = set(), []
    for v in base + joined + squashed:
        if v and v not in seen: seen.add(v); out.append(v)
    return out
need = set(LEX)
for r in ev106 + pk:
    for t in r['context_nouns'] + r.get('ours_nouns', []) + r['adjectives']: need.add(t)
need |= {w for s in list(need) for w in parts(s)}
PROBE = sorted({v for s in need for v in variants(s)}); print(f'{len(need)} strings -> {len(PROBE)} surface forms')

STATIC = {'glove': 'glove-wiki-gigaword-300', 'fasttext': 'fasttext-wiki-news-subwords-300', 'numberbatch': 'conceptnet-numberbatch-17-06-300'}
import gensim.downloader as api
for name, gname in STATIC.items():
    out = f'{ROOT}/models_cache/{name}_subset.pkl'
    if os.path.exists(out): print(f'{name}: cached {len(pickle.load(open(out, "rb")))} vectors'); continue
    t0 = time.time(); kv = api.load(gname)
    key = (lambda p: f'/c/en/{p.lower()}') if name == 'numberbatch' else (lambda p: p)
    VEC = {p: np.asarray(kv[key(p)], np.float32) for p in PROBE if key(p) in kv}
    if name == 'glove': VEC.update({p: np.asarray(kv[p.lower()], np.float32) for p in PROBE if p not in VEC and p.lower() in kv})   # glove is lowercase-only
    del kv; gc.collect(); pickle.dump(VEC, open(out, 'wb'), protocol=4)
    print(f'{name}: {len(VEC)} vectors cached ({os.path.getsize(out)/1e6:.1f} MB, {time.time()-t0:.0f}s)')

In [ ]:
# P2.2 — sentence-transformer term caches (all-mpnet-base-v2, bge-base-en-v1.5, multilingual-e5-base) on the T4; only the probe terms are encoded
import subprocess, sys, os, pickle, time, gc, numpy as np, torch
try: from sentence_transformers import SentenceTransformer
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True); from sentence_transformers import SentenceTransformer
TRANS = {'mpnet': ('sentence-transformers/all-mpnet-base-v2', ''), 'bge': ('BAAI/bge-base-en-v1.5', ''), 'e5': ('intfloat/multilingual-e5-base', 'query: ')}
TERMS = sorted(need)
for name, (mid, prefix) in TRANS.items():
    out = f'{ROOT}/models_cache/{name}_terms.pkl'
    if os.path.exists(out): print(f'{name}: cached {len(pickle.load(open(out, "rb")))} terms'); continue
    t0 = time.time(); m = SentenceTransformer(mid, device='cuda')
    E = m.encode([prefix + t.replace('-', ' ').replace('_', ' ') for t in TERMS], batch_size=256, normalize_embeddings=True, show_progress_bar=False)
    pickle.dump({t: E[i].astype(np.float32) for i, t in enumerate(TERMS)}, open(out, 'wb'), protocol=4)
    del m; gc.collect(); torch.cuda.empty_cache(); print(f'{name}: {len(TERMS)} terms x {E.shape[1]}d cached ({time.time()-t0:.0f}s)')

In [ ]:
# P2.3 — the ablation grid: 7 backbones x {nouns-only, + all adjectives (naive), + EN blend (ours, λ by 5-fold CV)} on the 106 and PK n>=3
import pickle, json, re, numpy as np
from collections import Counter
from sklearn.model_selection import KFold
def ladder(VEC):                       # static vectors: surface-variant ladder, then mean of parts (H2 recipe)
    def f(s):
        for v in variants(s):
            if v in VEC: return VEC[v]
        sub = [VEC[v] for w in parts(s) for v in variants(w) if v in VEC][:8]
        return np.mean(sub, 0) if sub else None
    return f
BACKBONES = {'W2V (GoogleNews)': ladder(pickle.load(open(f'{ROOT}/models_cache/w2v_googlenews_subset_v2.pkl', 'rb'))),
             'GloVe (wiki-gigaword)': ladder(pickle.load(open(f'{ROOT}/models_cache/glove_subset.pkl', 'rb'))),
             'fastText (wiki-news)': ladder(pickle.load(open(f'{ROOT}/models_cache/fasttext_subset.pkl', 'rb'))),
             'ConceptNet Numberbatch': ladder(pickle.load(open(f'{ROOT}/models_cache/numberbatch_subset.pkl', 'rb'))),
             'all-mpnet-base-v2': pickle.load(open(f'{ROOT}/models_cache/mpnet_terms.pkl', 'rb')).get,
             'bge-base-en-v1.5': pickle.load(open(f'{ROOT}/models_cache/bge_terms.pkl', 'rb')).get,
             'multilingual-e5-base': pickle.load(open(f'{ROOT}/models_cache/e5_terms.pkl', 'rb')).get}
enz = np.load(f'{ROOT}/results_i/i3_elasticnet.npz', allow_pickle=True); COEF, EN_CLS, EN_ADJ = enz['coef'], list(enz['classes']), list(enz['adj_vocab']); EN_AIX = {a: j for j, a in enumerate(EN_ADJ)}
def build(vec):
    TAGS = [t for t in LEX if vec(t) is not None]; M = np.vstack([vec(t) for t in TAGS]); M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9); IX = {t: i for i, t in enumerate(TAGS)}
    c2w = {c: IX[c] for c in EN_CLS if c in IX}
    def score(terms):
        s = np.zeros(len(TAGS), np.float32)
        for t in terms:
            v = vec(t)
            if v is None: continue
            sims = M @ (v / (np.linalg.norm(v) + 1e-9)); s += (sims - sims.min()) / (sims.max() - sims.min() + 1e-9)
        return s
    def en(adjs):
        s = np.zeros(len(TAGS), np.float32); rc = [COEF[:, EN_AIX[a]] for a in adjs if a in EN_AIX]
        if not rc: return s
        agg = np.sum(rc, 0)
        for c, w in c2w.items(): s[w] = agg[EN_CLS.index(c)]
        lo, hi = s.min(), s.max(); return (s - lo) / (hi - lo + 1e-9) if hi > lo else s
    return TAGS, IX, score, en
def ranks(rows, TAGS, IX, score, en, cond, lam=0.0):
    out = []
    for r in rows:
        terms = [t for t in r['context_nouns'] if t.lower() not in ABSTRACT] + (r['adjectives'] if cond == 'naive' else [])
        s = score(terms); s = (s - s.min()) / (s.max() - s.min() + 1e-9)
        if cond == 'en' and lam > 0: s = (1 - lam) * s + lam * en(r['adjectives'])
        g = r['gold_tag']; out.append(int((s >= s[IX[g]]).sum()) if g in IX else len(TAGS) + 1)
    return np.array(out, float)
def cv_en(rows, TAGS, IX, score, en):
    R = {L: ranks(rows, TAGS, IX, score, en, 'en', L) for L in LAMS}; held = np.zeros(len(rows)); ch = []
    for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(np.arange(len(rows))):
        L = max(LAMS, key=lambda l: (R[l][tr_i] <= 3).mean()); ch.append(L); held[te_i] = R[L][te_i]
    return held, ch
SETS = [('106', ev106, None), ('PK n>=3', pk, SUB)]
RES = {}; hdr = f'{"backbone":<24}{"cond":<14}' + ''.join(f'{s + " " + m:>16}' for s, _, _ in SETS for m in ('H@1', 'H@3', 'nDCG'))
print(hdr)
for bname, vec in BACKBONES.items():
    TAGS, IX, score, en = build(vec); RES[bname] = {'n_tags_embedded': len(TAGS)}
    for cond in ('nouns', 'naive', 'en'):
        row = {}
        for sname, rows, mask in SETS:
            rk, ch = (cv_en(rows, TAGS, IX, score, en) if cond == 'en' else (ranks(rows, TAGS, IX, score, en, cond), None))
            m = metrics(rk if mask is None else rk[mask]); row[sname] = {**m, 'cv_lambdas': ch}
        RES[bname][cond] = row
        print(f'{bname:<24}{ {"nouns": "nouns-only", "naive": "+adj naive", "en": "+EN blend"}[cond]:<14}' + ''.join(f'{row[s][k]:>16.3f}' for s, _, _ in SETS for k in ('Hits@1', 'Hits@3', 'nDCG@20')))
print('\nclaims check (Hits@3):')
for b in BACKBONES:
    for s, _, _ in SETS:
        n, a, e = RES[b]['nouns'][s]['Hits@3'], RES[b]['naive'][s]['Hits@3'], RES[b]['en'][s]['Hits@3']
        print(f'  {b:<24}{s:<9} EN-blend vs nouns {e-n:+.3f} {"✓" if e > n else "✗"} | naive-adj vs nouns {a-n:+.3f} {"hurts" if a < n else "no harm"}')
json.dump(RES, open(f'{ROOT}/results_p/p2_backbones.json', 'w'), indent=1); print('\nwrote results_p/p2_backbones.json')

In [ ]:
# P2.4 — classifier arm: DistilBERT fine-tuned on the weak-labelled Wikivoyage sentences (69 tags), ranked by softmax; plus the proposal's fusion (softmax + EN patterns)
import time, json, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import KFold
lab = pd.read_parquet(f'{ROOT}/results_m/weak_labelled_v2.parquet', columns=['text', 'weak_tag', 'split'])
trn = lab[lab.split == 'train'].reset_index(drop=True)
keep = (trn.groupby('weak_tag', group_keys=False).apply(lambda g: g.sample(min(len(g), max(1, round(len(g) * 150_000 / len(trn)))), random_state=42)).index.values)
CLS = sorted(trn.weak_tag.unique()); CIX = {c: i for i, c in enumerate(CLS)}
sub = trn.loc[keep].sample(frac=1, random_state=42); texts, ys = sub.text.tolist(), np.array([CIX[t] for t in sub.weak_tag])
tok = AutoTokenizer.from_pretrained('distilbert-base-uncased'); dev = 'cuda'
mdl = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=len(CLS)).to(dev)
opt = torch.optim.AdamW(mdl.parameters(), lr=5e-5, weight_decay=0.01); scaler = torch.cuda.amp.GradScaler()
BS, MAXLEN = 64, 64; steps = len(texts) // BS; sched = torch.optim.lr_scheduler.LinearLR(opt, 1.0, 0.0, total_iters=steps)
t0 = time.time(); mdl.train()
for i in range(steps):
    enc = tok(texts[i*BS:(i+1)*BS], padding=True, truncation=True, max_length=MAXLEN, return_tensors='pt').to(dev)
    with torch.autocast('cuda', dtype=torch.float16): loss = mdl(**enc, labels=torch.tensor(ys[i*BS:(i+1)*BS], device=dev)).loss
    opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    if i % 500 == 0: print(f'  step {i}/{steps} loss {loss.item():.3f} ({time.time()-t0:.0f}s)')
train_time = time.time() - t0; mdl.eval()
def probs(rows):
    out = []
    with torch.no_grad():
        for i in range(0, len(rows), 128):
            enc = tok([r['sentence'] for r in rows[i:i+128]], padding=True, truncation=True, max_length=128, return_tensors='pt').to(dev)
            out.append(torch.softmax(mdl(**enc).logits.float(), -1).cpu().numpy())
    return np.vstack(out)
# dev-split accuracy on the weak labels (sanity), then eval on the gold sets
dv = lab[lab.split == 'dev'].sample(5000, random_state=42); pdv = probs([{'sentence': t} for t in dv.text]); dev_acc = float((np.array(CLS)[pdv.argmax(1)] == dv.weak_tag.values).mean())
print(f'DistilBERT: {len(texts):,} rows, 1 epoch, {train_time:.0f}s | weak-label dev top-1 acc {dev_acc:.3f} (69 classes)')
def cls_ranks(rows, P, lam=0.0, en=None, TAGS=None, IX=None):
    out = []
    for r, p in zip(rows, P):
        if lam > 0:   # fusion: classifier softmax mapped onto the ranker tag space, blended with the EN pattern score
            s = np.zeros(len(TAGS), np.float32)
            for c, k in CIX.items():
                if c in IX: s[IX[c]] = p[k]
            s = (s - s.min()) / (s.max() - s.min() + 1e-9); s = (1 - lam) * s + lam * en(r['adjectives'])
            g = r['gold_tag']; out.append(int((s >= s[IX[g]]).sum()) if g in IX else len(TAGS) + 1)
        else:
            g = r['gold_tag']; out.append(int((p >= p[CIX[g]]).sum()) if g in CIX else len(CLS) + 1)
    return np.array(out, float)
W2 = build(BACKBONES['W2V (GoogleNews)']); TAGSw, IXw, _, enw = W2
RESC = {}
for sname, rows, mask in SETS:
    P = probs(rows); r0 = cls_ranks(rows, P)
    R = {L: cls_ranks(rows, P, L, enw, TAGSw, IXw) for L in LAMS}; held = np.zeros(len(rows)); ch = []
    for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(np.arange(len(rows))):
        L = max(LAMS, key=lambda l: (R[l][tr_i] <= 3).mean()); ch.append(L); held[te_i] = R[L][te_i]
    m0, m1 = metrics(r0 if mask is None else r0[mask]), metrics(held if mask is None else held[mask])
    cov = np.mean([r['gold_tag'] in CIX for r in rows]) if mask is None else np.mean([r['gold_tag'] in CIX for r, k in zip(rows, mask) if k])
    print(f'{sname:<9} gold tags inside the 69 classes: {cov:.0%} | classifier-only  H@1 {m0["Hits@1"]:.3f} H@3 {m0["Hits@3"]:.3f} H@5 {m0["Hits@5"]:.3f} nDCG {m0["nDCG@20"]:.3f} | + EN fusion (λ{ch}) H@1 {m1["Hits@1"]:.3f} H@3 {m1["Hits@3"]:.3f} nDCG {m1["nDCG@20"]:.3f}')
    RESC[sname] = {'gold_coverage_69': float(cov), 'classifier_only': m0, 'classifier_plus_en_cv': m1, 'cv_lambdas': ch}
json.dump({'train_rows': len(texts), 'epochs': 1, 'train_time_s': train_time, 'dev_acc_weak': dev_acc, 'classes': len(CLS), **RESC}, open(f'{ROOT}/results_p/p2_classifier.json', 'w'), indent=1)
mdl.save_pretrained(f'{ROOT}/models_cache/distilbert_weak69'); tok.save_pretrained(f'{ROOT}/models_cache/distilbert_weak69'); print('wrote results_p/p2_classifier.json, models_cache/distilbert_weak69')

In [ ]:
# P2.5 — pattern-granularity arm: frequentist EN refit on {adjective 1-itemsets} / {adjective-noun 2-itemsets} / {both}, same 120k stratified subsample, W2V ranker blend
import time, json, numpy as np
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
Xtr = sparse.load_npz(f'{ROOT}/results_p/Xtr.npz'); meta = np.load(f'{ROOT}/results_p/p1_feat_meta.npz', allow_pickle=True)
PV, CLASSES, ytr, NADJ = list(meta['patvocab']), list(meta['classes']), meta['ytr'], int(meta['n_adj']); P = len(PV)
rng = np.random.default_rng(42); keep = np.concatenate([rng.choice(np.where(ytr == c)[0], min((ytr == c).sum(), max(1, round((ytr == c).mean() * 120_000))), replace=False) for c in range(len(CLASSES))])
FAM = {'adjectives only (1-itemsets)': np.arange(NADJ), 'adj-noun pairs only (2-itemsets)': np.arange(NADJ, P), 'both (P1.3)': np.arange(P)}
ADJ_SET, PAIR_SET = set(PV[:NADJ]), set(PV[NADJ:]); PIX = {p: j for j, p in enumerate(PV)}
def pat_feats(r): return [PIX[a] for a in r['adjectives'] if a in ADJ_SET] + [PIX[p] for p in r['transaction'] if p in PAIR_SET]
TAGSw, IXw, scorew, _ = build(BACKBONES['W2V (GoogleNews)']); c2w = {c: IXw[c] for c in CLASSES if c in IXw}; cix = {c: k for k, c in enumerate(CLASSES)}
def make_pen(coefPC, cols):
    colset = set(cols.tolist())
    def en(r):
        s = np.zeros(len(TAGSw), np.float32); f = [j for j in pat_feats(r) if j in colset]
        if not f: return s
        agg = coefPC[f].sum(0)
        for c, w in c2w.items(): s[w] = agg[cix[c]]
        lo, hi = s.min(), s.max(); return (s - lo) / (hi - lo + 1e-9) if hi > lo else s
    return en
def rk(rows, lam, en):
    out = []
    for r in rows:
        s = scorew([t for t in r['context_nouns'] if t.lower() not in ABSTRACT]); s = (s - s.min()) / (s.max() - s.min() + 1e-9)
        if lam > 0: s = (1 - lam) * s + lam * en(r)
        g = r['gold_tag']; out.append(int((s >= s[IXw[g]]).sum()) if g in IXw else len(TAGSw) + 1)
    return np.array(out, float)
RESG = {}
print(f'{"family":<34}{"fit":>6}{"nnz":>6}' + ''.join(f'{s + " " + m:>14}' for s, _, _ in SETS for m in ('cov', 'H@1', 'H@3', 'nDCG')))
for fam, cols in FAM.items():
    t0 = time.time(); en_ = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.5, max_iter=200, tol=1e-3, n_jobs=-1).fit(Xtr[keep][:, cols], ytr[keep])
    coefPC = np.zeros((P, len(CLASSES)), np.float32); coefPC[cols] = en_.coef_.T; ft = time.time() - t0; en = make_pen(coefPC, cols); row = {'fit_s': ft, 'pct_nonzero': float((en_.coef_ != 0).mean())}
    for sname, rows, mask in SETS:
        cov = np.mean([any(j in set(cols.tolist()) for j in pat_feats(r)) for r, k in zip(rows, (mask if mask is not None else [True]*len(rows))) if k])
        R = {L: rk(rows, L, en) for L in LAMS}; held = np.zeros(len(rows)); ch = []
        for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(np.arange(len(rows))):
            L = max(LAMS, key=lambda l: (R[l][tr_i] <= 3).mean()); ch.append(L); held[te_i] = R[L][te_i]
        m = metrics(held if mask is None else held[mask]); row[sname] = {'coverage': float(cov), **m, 'cv_lambdas': ch}
    RESG[fam] = row
    print(f'{fam:<34}{ft:>5.0f}s{row["pct_nonzero"]:>6.0%}' + ''.join(f'{row[s]["coverage"]:>14.0%}{row[s]["Hits@1"]:>14.3f}{row[s]["Hits@3"]:>14.3f}{row[s]["nDCG@20"]:>14.3f}' for s, _, _ in SETS))
json.dump(RESG, open(f'{ROOT}/results_p/p2_granularity.json', 'w'), indent=1); print('wrote results_p/p2_granularity.json')

In [ ]:
# P2.6 — leakage check: are any 106 / PK eval sentences (or near-duplicates) inside the training corpora?
import re, json, numpy as np, pandas as pd, pyarrow.parquet as pq
from collections import Counter, defaultdict
norm = lambda s: re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]+', ' ', s.lower())).strip()
toks = lambda s: set(norm(s).split())
def corpus_texts(path):
    cols = pq.ParquetFile(path).schema.names; col = 'text' if 'text' in cols else ('sentence' if 'sentence' in cols else cols[0])
    return pd.read_parquet(path, columns=[col])[col].astype(str).tolist()
CORPORA = {'weak_labelled_v2 (EN/FP-Growth training)': corpus_texts(f'{ROOT}/results_m/weak_labelled_v2.parquet'),
           'annotated_v2 (full 1M corpus)': corpus_texts(f'{ROOT}/annotated/annotated_v2.parquet')}
EVALS = {'106 (base paper)': [r['sentence'] for r in ev106], 'PK 199': [r['sentence'] for r in pk]}
report = {}
for cname, texts in CORPORA.items():
    exact = set(); inv = defaultdict(list); df = Counter()
    for i, t in enumerate(texts):
        n = norm(t); exact.add(n); ts = set(n.split()); df.update(ts)
        for w in ts: inv[w].append(i)
    for ename, sents in EVALS.items():
        hits_exact, hits_near = [], []
        for k, s in enumerate(sents):
            n = norm(s); ts = set(n.split())
            if n in exact: hits_exact.append(k); continue
            rare = sorted(ts, key=lambda w: df.get(w, 0))[:3]
            cands = set().union(*[inv[w] for w in rare if df.get(w, 0) < 5000])
            best = max((len(ts & toks(texts[i])) / len(ts | toks(texts[i])) for i in cands), default=0.0)
            if best >= 0.6: hits_near.append((k, round(best, 2)))
        report[f'{ename} vs {cname}'] = {'exact': hits_exact, 'near_dup_jaccard>=0.6': hits_near}
        print(f'{ename:<18} vs {cname:<40} exact {len(hits_exact):>3} | near-dup (token Jaccard >= 0.6) {len(hits_near):>3}   {hits_near[:6]}')
json.dump(report, open(f'{ROOT}/results_p/p2_leakage_check.json', 'w'), indent=1); print('wrote results_p/p2_leakage_check.json')

In [ ]:
# P2.6b — PK headline with the 5 near-duplicate rows excluded (leakage-robust numbers)
leak = {k for k, _ in json.load(open(f'{ROOT}/results_p/p2_leakage_check.json'))['PK 199 vs weak_labelled_v2 (EN/FP-Growth training)']['near_dup_jaccard>=0.6']}
clean = np.array([i not in leak for i in range(len(pk))]); print('excluded rows:', sorted(pk[i]['row'] for i in leak), '| remaining', clean.sum(), '| n>=3 & clean', (clean & SUB).sum())
for name, key in [('W2V (Alg.2)', 'W2V (Alg.2)'), ('transfer v1, λ=0.35', 'transfer v1, λ=0.35 fixed')]:
    for lbl, m in [('n>=3 all', SUB), ('n>=3 minus near-dups', SUB & clean)]:
        r = metrics(RANKS[key][m]); print(f'  {name:<22} {lbl:<22} H@1 {r["Hits@1"]:.3f}  H@3 {r["Hits@3"]:.3f}  H@5 {r["Hits@5"]:.3f}  nDCG {r["nDCG@20"]:.3f}')
json.dump({'excluded_rows': sorted(pk[i]['row'] for i in leak), 'n_clean_n3': int((clean & SUB).sum()),
           'W2V_clean': metrics(RANKS['W2V (Alg.2)'][SUB & clean]), 'transfer_clean': metrics(RANKS['transfer v1, λ=0.35 fixed'][SUB & clean])},
          open(f'{ROOT}/results_m/m5_leakage_robust.json', 'w'), indent=1); print('wrote results_m/m5_leakage_robust.json')